<a href="https://colab.research.google.com/github/minhduy23127/minhduy23127.github.io/blob/main/TrainCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cài đặt công cụ Kaggle
!pip install -q kaggle

import os
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write('KGAT_db46314c5d56d91c15a34a3672abc898')

# Tải tập dữ liệu fer2013 từ Kaggle về Colab
!kaggle datasets download -d msambare/fer2013

# Giải nén tập dữ liệu
!unzip -q fer2013.zip -d .
print("Tải và giải nén dữ liệu thành công!")

Dataset URL: https://www.kaggle.com/datasets/msambare/fer2013
License(s): DbCL-1.0
100% 60.3M/60.3M [00:03<00:00, 16.7MB/s]

Tải và giải nén dữ liệu thành công!


# Huấn luyện CNN nhận diện cảm xúc khuôn mặt (tự cài đặt các block của CNN)

Notebook huấn luyện một mạng CNN phân loại ảnh khuôn mặt xám 48×48 thành 7 lớp cảm xúc, chia làm 3 phần:

1. Chuẩn bị dữ liệu
2. Kiến trúc CNN
3. Huấn luyện

Vì đồ án nghiên cứu lan truyền ngược trong CNN nên các block của CNN (Conv, ReLU, BatchNorm, MaxPool, Dropout, Flatten, Dense) được **tự viết cả lan truyền tiến lẫn lan truyền ngược**, không dùng các lớp có sẵn của Keras. Những phần không thuộc CNN thì dùng thư viện.

| Thành phần | Cách làm |
|---|---|
| Conv2D, ReLU, BatchNorm, MaxPool, Dropout, Flatten, Dense (tiến và ngược) | Tự cài đặt (Phần 2) |
| Đọc ảnh, chia train/validation, tăng cường dữ liệu | Thư viện (`ImageDataGenerator`) |
| Softmax, hàm loss Cross-Entropy | Thư viện |
| Thuật toán tối ưu Adam | Thư viện |
| Lưu mô hình tốt nhất, giảm learning rate | Thư viện (callbacks) |

## Phần 1. Chuẩn bị dữ liệu

Dữ liệu gồm hai thư mục `train/` và `test/`. Trong mỗi thư mục có 7 thư mục con, mỗi thư mục con là một cảm xúc và chứa ảnh xám 48×48. Thư mục `train/` được chia 80% để huấn luyện và 20% để kiểm tra trong lúc huấn luyện (validation). Thư mục `test/` không dùng để huấn luyện hay chọn mô hình mà chỉ dùng ở Phần 4 để đánh giá cuối cùng trên dữ liệu chưa từng thấy.

Đầu tiên là import thư viện.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

### Đọc ảnh và tăng cường dữ liệu

`ImageDataGenerator` đọc ảnh từ thư mục và biến đổi ngẫu nhiên mỗi lần lấy ảnh ra để mô hình thấy nhiều biến thể của cùng một ảnh:

| Tham số | Ý nghĩa |
|---|---|
| `rescale=1./255` | Đưa pixel từ [0, 255] về [0, 1]. Việc này quan trọng với lan truyền ngược vì gradient của kernel tỉ lệ với giá trị pixel |
| `rotation_range=15` | Xoay ngẫu nhiên tối đa ±15 độ |
| `width_shift_range=0.1` | Dịch ngang ngẫu nhiên tối đa 10% |
| `height_shift_range=0.1` | Dịch dọc ngẫu nhiên tối đa 10% |
| `horizontal_flip=True` | Lật ngang ngẫu nhiên |
| `validation_split=0.2` | Lấy 20% ảnh của `train/` làm validation |

`flow_from_directory` tạo ra các batch 64 ảnh cỡ 48×48, một kênh xám, nhãn dạng one-hot (7 lớp). `subset='training'` lấy 80% ảnh để học, `subset='validation'` lấy 20% còn lại. Giống file gốc, hai bộ đọc dùng chung `train_datagen` nên ảnh validation cũng bị xoay, dịch, lật ngẫu nhiên.

In [ ]:
train_dir = 'train'
test_dir = 'test'      # dùng ở Phần 4 để đánh giá

train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(48, 48),
    color_mode='grayscale',
    batch_size=64,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(48, 48),
    color_mode='grayscale',
    batch_size=64,
    class_mode='categorical',
    subset='validation'
)

Found 22968 images belonging to 7 classes.
Found 5741 images belonging to 7 classes.


## Phần 2. Kiến trúc CNN

Mạng gồm 3 khối tích chập để trích đặc trưng, sau đó là phần Fully Connected để phân loại:

**Conv → ReLU → BatchNorm → MaxPool → Dropout** (lặp 3 lần với 64, 128, 256 kernel) → **Flatten → Dense(256) → ReLU → BatchNorm → Dropout → Dense(7) → Softmax**

### Cách tự cài đặt mỗi block

Mỗi block được viết thành một hàm `op(x, tham số)` có gắn `@tf.custom_gradient`. Hàm này trả về hai thứ:
- đầu ra `y`: đây là **lan truyền tiến**;
- một hàm `grad(dy)`: đây là **lan truyền ngược**. Nó nhận `dy` = ∂L/∂y (gradient của loss theo đầu ra, do block phía sau truyền về) và trả về ∂L/∂x (để truyền tiếp cho block phía trước) cùng ∂L/∂(tham số) nếu block có tham số.

Nhờ `@tf.custom_gradient`, TensorFlow gọi đúng hàm `grad` tự viết khi lan truyền ngược, không tự suy đạo hàm của các block này. Mỗi block lại được bọc trong một lớp `Layer` chỉ để chứa trọng số và nối vào `Sequential`. Thư viện chỉ dùng cho các phép toán tensor cơ bản (cộng, nhân ma trận, reshape, pad...), khung `Layer`, bộ khởi tạo trọng số và lớp Softmax.

Quy ước kích thước: ảnh `(N, H, W, C)` (số ảnh, cao, rộng, kênh), kernel `(K, K, C, F)`.

### 2.1. Conv2D

Đầu vào $x$ có kích thước $(N,H,W,C)$, kernel $W$ có kích thước $(K,K,C,F)$, bias $b$ có $F$ phần tử. Ở đây $K=3$, stride 1, padding `same` (thêm viền 0 dày 1 để $H, W$ không đổi).

**Lan truyền tiến:**

$$y_{n,i,j,f} = b_f + \sum_{u=0}^{K-1}\sum_{v=0}^{K-1}\sum_{c=1}^{C} W_{u,v,c,f}\; x^{pad}_{n,\,i+u,\,j+v,\,c}$$

Trong code dùng kỹ thuật **im2col**: với mỗi vị trí $(i,j)$, xếp cửa sổ $K\times K\times C$ của ảnh thành một vector dài $K^2C$ để được `cols` có kích thước $(N,H,W,K^2C)$. Kernel được trải thành ma trận $W_{mat}$ kích thước $(K^2C, F)$. Khi đó phép tích chập chỉ còn là một phép nhân ma trận:

$$y = \text{cols}\cdot W_{mat} + b$$

**Lan truyền ngược:** nhận $dy=\partial L/\partial y$ (cùng kích thước với $y$):

$$\frac{\partial L}{\partial b_f} = \sum_{n,i,j} dy_{n,i,j,f}$$

$$\frac{\partial L}{\partial W_{u,v,c,f}} = \sum_{n,i,j} dy_{n,i,j,f}\; x^{pad}_{n,\,i+u,\,j+v,\,c} \quad\Longleftrightarrow\quad \frac{\partial L}{\partial W_{mat}} = \text{cols}^{T}\cdot dy$$

$$\frac{\partial L}{\partial \text{cols}} = dy\cdot W_{mat}^{T}, \qquad \frac{\partial L}{\partial x} = \text{col2im}\left(\frac{\partial L}{\partial \text{cols}}\right)$$

**Loss truyền về một kernel như thế nào:** mỗi trọng số $W_{u,v,c,f}$ được dùng ở mọi vị trí $(i,j)$ của ảnh (chia sẻ trọng số), nên gradient của nó là tổng đóng góp của mọi vị trí. Tại mỗi vị trí lấy $dy$ nhân với đúng pixel mà trọng số đó đã nhân lúc tiến.

**Loss truyền về ảnh đầu vào:** một pixel nằm trong nhiều cửa sổ chồng nhau nên gradient của nó là tổng gradient từ tất cả các cửa sổ chứa nó. `col2im` làm việc cộng dồn này: đặt gradient của từng cửa sổ về đúng vị trí rồi cộng lại, sau đó bỏ phần viền đã thêm.

In [ ]:
def im2col(x, k):
    """Biến mỗi cửa sổ k x k thành một vector: (N,H,W,C) -> (N,H,W,k*k*C)"""
    p = k // 2
    xp = tf.pad(x, [[0, 0], [p, p], [p, p], [0, 0]])               # padding same
    H, W = x.shape[1], x.shape[2]
    patches = [xp[:, u:u + H, v:v + W, :] for u in range(k) for v in range(k)]
    return tf.concat(patches, axis=-1)


def col2im(dcols, k, H, W, C):
    """Ngược lại im2col, cộng gradient của các cửa sổ chồng nhau về từng pixel"""
    p = k // 2
    d = tf.reshape(dcols, [-1, H, W, k * k, C])
    dxp, i = None, 0
    for u in range(k):
        for v in range(k):
            part = tf.pad(d[:, :, :, i, :], [[0, 0], [u, k - 1 - u], [v, k - 1 - v], [0, 0]])
            dxp = part if dxp is None else dxp + part
            i += 1
    return dxp[:, p:p + H, p:p + W, :]                              # bỏ viền


@tf.custom_gradient
def conv2d_op(x, W, b):
    K, C, F = W.shape[0], W.shape[2], W.shape[3]
    H, Wd = x.shape[1], x.shape[2]

    # lan truyền tiến
    cols = im2col(x, K)                                             # (N,H,W,K*K*C)
    W_mat = tf.reshape(W, [K * K * C, F])                           # trải kernel thành ma trận
    y = tf.tensordot(cols, W_mat, axes=1) + b                       # (N,H,W,F)

    # lan truyền ngược
    def grad(dy):
        dW_mat = tf.tensordot(cols, dy, axes=[[0, 1, 2], [0, 1, 2]])   # dL/dW = cols^T . dy
        db = tf.reduce_sum(dy, axis=[0, 1, 2])                          # dL/db
        dcols = tf.tensordot(dy, W_mat, axes=[[3], [1]])                # dL/dcols = dy . W_mat^T
        dx = col2im(dcols, K, H, Wd, C)                                 # dL/dx
        return dx, tf.reshape(dW_mat, [K, K, C, F]), db

    return y, grad


class Conv2DLayer(tf.keras.layers.Layer):
    def __init__(self, filters, kernel_size=3, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        C = input_shape[-1]
        k = self.kernel_size
        self.W = self.add_weight(name='W', shape=(k, k, C, self.filters), initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='b', shape=(self.filters,), initializer='zeros', trainable=True)

    def call(self, x):
        return conv2d_op(x, self.W, self.b)

    def compute_output_shape(self, input_shape):
        return tuple(input_shape[:3]) + (self.filters,)

    def get_config(self):
        config = super().get_config()
        config.update({'filters': self.filters, 'kernel_size': self.kernel_size})
        return config

### 2.2. ReLU

**Tiến:** $y = \max(0, x)$.

**Ngược:** chỗ nào $x\le 0$ thì đầu ra không phụ thuộc vào $x$ nên không có gradient đi qua:

$$\frac{\partial L}{\partial x} = dy\odot \mathbf{1}[x>0]$$

Trong mạng này ReLU đứng ngay sau Conv và sau Dense (rồi mới đến BatchNorm), giống file gốc.

In [ ]:
@tf.custom_gradient
def relu_op(x):
    mask = tf.cast(x > 0, x.dtype)                  # lưu vị trí x > 0
    y = x * mask                                    # tiến

    def grad(dy):
        return dy * mask                            # ngược: chỉ truyền gradient ở chỗ x > 0

    return y, grad


class ReLULayer(tf.keras.layers.Layer):
    def call(self, x):
        return relu_op(x)

    def compute_output_shape(self, input_shape):
        return input_shape

### 2.3. BatchNorm

Chuẩn hóa theo từng kênh. Với một kênh có $m$ phần tử trong batch ($m=N\cdot H\cdot W$ sau lớp Conv, $m=N$ sau lớp Dense):

**Tiến (lúc huấn luyện):**

$$\mu=\frac1m\sum x,\quad \sigma^2=\frac1m\sum (x-\mu)^2,\quad \hat x=\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}},\quad y=\gamma\hat x+\beta$$

Ngoài ra $\mu$, $\sigma^2$ được lưu thành trung bình trượt (momentum 0.99). Lúc dự đoán không còn batch để tính nên dùng trung bình trượt này thay cho $\mu$, $\sigma^2$.

**Ngược:** đặt $\hat\delta = dy\cdot\gamma$, các tổng lấy trên $m$ phần tử của kênh:

$$\frac{\partial L}{\partial\gamma}=\sum dy\,\hat x,\qquad \frac{\partial L}{\partial\beta}=\sum dy$$

$$\frac{\partial L}{\partial x}=\frac{1}{m\sqrt{\sigma^2+\epsilon}}\Big(m\hat\delta-\sum\hat\delta-\hat x\sum\hat\delta\,\hat x\Big)$$

Công thức của $\partial L/\partial x$ dài hơn các block khác vì $\mu$ và $\sigma^2$ cũng phụ thuộc vào $x$. Trong code, `mean` và `var` được tính bên ngoài rồi truyền vào `batchnorm_op` như hằng số, nhưng công thức $\partial L/\partial x$ ở trên đã tính cả phần phụ thuộc đó nên kết quả vẫn đúng.

In [ ]:
@tf.custom_gradient
def batchnorm_op(x, gamma, beta, mean, var, eps):
    axes = list(range(len(x.shape) - 1))            # các trục trừ trục kênh
    inv_std = tf.math.rsqrt(var + eps)

    # lan truyền tiến
    xhat = (x - mean) * inv_std
    y = gamma * xhat + beta

    # lan truyền ngược
    def grad(dy):
        m = tf.cast(tf.size(x), x.dtype) / tf.cast(tf.size(gamma), x.dtype)    # số phần tử mỗi kênh
        dgamma = tf.reduce_sum(dy * xhat, axis=axes)
        dbeta = tf.reduce_sum(dy, axis=axes)
        dxhat = dy * gamma
        dx = inv_std / m * (m * dxhat
                            - tf.reduce_sum(dxhat, axis=axes)
                            - xhat * tf.reduce_sum(dxhat * xhat, axis=axes))
        return dx, dgamma, dbeta, None, None, None  # mean, var, eps không cần gradient

    return y, grad


class BatchNormLayer(tf.keras.layers.Layer):
    def __init__(self, momentum=0.99, epsilon=1e-3, **kwargs):
        super().__init__(**kwargs)
        self.momentum = momentum
        self.epsilon = epsilon

    def build(self, input_shape):
        C = input_shape[-1]
        self.gamma = self.add_weight(name='gamma', shape=(C,), initializer='ones', trainable=True)
        self.beta = self.add_weight(name='beta', shape=(C,), initializer='zeros', trainable=True)
        self.moving_mean = self.add_weight(name='moving_mean', shape=(C,), initializer='zeros', trainable=False)
        self.moving_var = self.add_weight(name='moving_var', shape=(C,), initializer='ones', trainable=False)

    def call(self, x, training=None):
        if training:
            axes = list(range(len(x.shape) - 1))
            mean = tf.reduce_mean(x, axis=axes)
            var = tf.reduce_mean(tf.square(x - mean), axis=axes)
            # cập nhật trung bình trượt
            self.moving_mean.assign(self.momentum * self.moving_mean + (1 - self.momentum) * mean)
            self.moving_var.assign(self.momentum * self.moving_var + (1 - self.momentum) * var)
            return batchnorm_op(x, self.gamma, self.beta, mean, var, self.epsilon)
        # lúc dự đoán dùng trung bình trượt
        return self.gamma * (x - self.moving_mean) * tf.math.rsqrt(self.moving_var + self.epsilon) + self.beta

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        config = super().get_config()
        config.update({'momentum': self.momentum, 'epsilon': self.epsilon})
        return config

### 2.4. MaxPooling 2×2

**Tiến:** chia ảnh thành các cửa sổ 2×2 không chồng nhau và lấy giá trị lớn nhất, nên $H$ và $W$ giảm một nửa. Đồng thời nhớ vị trí lớn nhất (`argmax`) trong mỗi cửa sổ.

**Ngược:** trong mỗi cửa sổ chỉ phần tử lớn nhất ảnh hưởng đến đầu ra, nên toàn bộ $dy$ được chuyển về đúng vị trí đó, 3 vị trí còn lại nhận gradient bằng 0.

In [ ]:
@tf.custom_gradient
def maxpool_op(x):
    H, W, C = x.shape[1], x.shape[2], x.shape[3]

    # lan truyền tiến: gom mỗi cửa sổ 2x2 rồi lấy max
    xr = tf.reshape(x, [-1, H // 2, 2, W // 2, 2, C])
    xr = tf.transpose(xr, [0, 1, 3, 5, 2, 4])                       # (N,H/2,W/2,C,2,2)
    xr = tf.reshape(xr, [-1, H // 2, W // 2, C, 4])
    y = tf.reduce_max(xr, axis=-1)
    idx = tf.argmax(xr, axis=-1)                                    # vị trí max trong mỗi cửa sổ

    # lan truyền ngược: dy chỉ đi về vị trí max
    def grad(dy):
        onehot = tf.one_hot(idx, 4, dtype=dy.dtype)                 # (N,H/2,W/2,C,4)
        d = tf.expand_dims(dy, -1) * onehot
        d = tf.reshape(d, [-1, H // 2, W // 2, C, 2, 2])
        d = tf.transpose(d, [0, 1, 4, 2, 5, 3])                     # (N,H/2,2,W/2,2,C)
        return tf.reshape(d, [-1, H, W, C])

    return y, grad


class MaxPool2DLayer(tf.keras.layers.Layer):
    def call(self, x):
        return maxpool_op(x)

    def compute_output_shape(self, input_shape):
        n, h, w, c = input_shape
        return (n, h // 2, w // 2, c)

### 2.5. Dropout

**Tiến (lúc huấn luyện):** tắt ngẫu nhiên mỗi phần tử với xác suất $r$ rồi chia cho $(1-r)$ để giá trị kỳ vọng không đổi: $y = x\odot m$, với $m_k\in\{0,\ 1/(1-r)\}$. Lúc dự đoán thì giữ nguyên $y=x$.

**Ngược:** phần tử bị tắt không ảnh hưởng đến loss nên không nhận gradient; các phần tử còn lại nhân với đúng mặt nạ đã dùng lúc tiến:

$$\frac{\partial L}{\partial x} = dy\odot m$$

In [ ]:
@tf.custom_gradient
def dropout_op(x, mask):
    y = x * mask                                    # tiến: nhân với mặt nạ

    def grad(dy):
        return dy * mask, None                      # ngược: nhân với mặt nạ

    return y, grad


class DropoutLayer(tf.keras.layers.Layer):
    def __init__(self, rate, **kwargs):
        super().__init__(**kwargs)
        self.rate = rate

    def call(self, x, training=None):
        if not training or self.rate == 0:          # lúc dự đoán thì giữ nguyên
            return x
        mask = tf.cast(tf.random.uniform(tf.shape(x)) >= self.rate, x.dtype) / (1.0 - self.rate)
        return dropout_op(x, mask)

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        config = super().get_config()
        config.update({'rate': self.rate})
        return config

### 2.6. Flatten và 2.7. Dense

**Flatten:** trải tensor $(N,H,W,C)$ thành ma trận $(N, HWC)$. Phép này chỉ đổi hình dạng nên lan truyền ngược chỉ cần `reshape` ngược lại.

**Dense:** $y = xW + b$ với $x$ kích thước $(N, n_{in})$ và $W$ kích thước $(n_{in}, n_{out})$.

$$\frac{\partial L}{\partial W} = x^{T}\,dy,\qquad \frac{\partial L}{\partial b}=\sum_n dy_n,\qquad \frac{\partial L}{\partial x}=dy\,W^{T}$$

In [ ]:
@tf.custom_gradient
def flatten_op(x):
    shape = list(x.shape[1:])
    size = 1
    for d in shape:
        size *= d

    def grad(dy):
        return tf.reshape(dy, [-1] + shape)         # ngược: reshape lại

    return tf.reshape(x, [-1, size]), grad


class FlattenLayer(tf.keras.layers.Layer):
    def call(self, x):
        return flatten_op(x)

    def compute_output_shape(self, input_shape):
        size = 1
        for d in input_shape[1:]:
            size *= d
        return (input_shape[0], size)


@tf.custom_gradient
def dense_op(x, W, b):
    y = tf.matmul(x, W) + b                         # tiến: y = xW + b

    def grad(dy):
        dx = tf.matmul(dy, W, transpose_b=True)     # dL/dx = dy . W^T
        dW = tf.matmul(x, dy, transpose_a=True)     # dL/dW = x^T . dy
        db = tf.reduce_sum(dy, axis=0)              # dL/db
        return dx, dW, db

    return y, grad


class DenseLayer(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        self.W = self.add_weight(name='W', shape=(input_shape[-1], self.units), initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='b', shape=(self.units,), initializer='zeros', trainable=True)

    def call(self, x):
        return dense_op(x, self.W, self.b)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.units)

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units})
        return config

### 2.8. Ghép thành mô hình

Ghép các block theo đúng kiến trúc của file gốc. Trọng số được khởi tạo bằng Glorot uniform, bias bằng 0 (BatchNorm: $\gamma=1$, $\beta=0$), cũng giống mặc định của Keras trong file gốc. Lớp Softmax cuối dùng thư viện vì không thuộc CNN.

| Block | Kích thước đầu ra | Số tham số huấn luyện |
|---|---|---|
| Đầu vào | 48×48×1 | 0 |
| Conv2D(64) + ReLU + BatchNorm | 48×48×64 | 640 + 128 |
| MaxPool + Dropout(0.25) | 24×24×64 | 0 |
| Conv2D(128) + ReLU + BatchNorm | 24×24×128 | 73,856 + 256 |
| MaxPool + Dropout(0.25) | 12×12×128 | 0 |
| Conv2D(256) + ReLU + BatchNorm | 12×12×256 | 295,168 + 512 |
| MaxPool + Dropout(0.25) | 6×6×256 | 0 |
| Flatten | 9216 | 0 |
| Dense(256) + ReLU + BatchNorm | 256 | 2,359,552 + 512 |
| Dropout(0.5) | 256 | 0 |
| Dense(7) + Softmax | 7 | 1,799 |

Tổng cộng 2,732,423 tham số huấn luyện, cộng thêm 1,408 giá trị trung bình trượt của BatchNorm (không huấn luyện bằng gradient).

In [ ]:
model = Sequential([
    tf.keras.Input(shape=(48, 48, 1)),

    # Khối 1
    Conv2DLayer(64), ReLULayer(), BatchNormLayer(), MaxPool2DLayer(), DropoutLayer(0.25),

    # Khối 2
    Conv2DLayer(128), ReLULayer(), BatchNormLayer(), MaxPool2DLayer(), DropoutLayer(0.25),

    # Khối 3
    Conv2DLayer(256), ReLULayer(), BatchNormLayer(), MaxPool2DLayer(), DropoutLayer(0.25),

    # Phần Fully Connected
    FlattenLayer(),
    DenseLayer(256), ReLULayer(), BatchNormLayer(), DropoutLayer(0.5),
    DenseLayer(7),
    tf.keras.layers.Softmax(),                      # softmax dùng thư viện
])

## Phần 3. Huấn luyện

### 3.1. Biên dịch mô hình

Hàm loss và thuật toán tối ưu không thuộc CNN nên dùng bản có sẵn của thư viện:
- **Loss:** Categorical Cross-Entropy, $L=-\frac1N\sum_i\sum_c y_{i,c}\ln\hat y_{i,c}$.
- **Optimizer:** Adam, learning rate ban đầu 0.001.
- **Metric:** accuracy.

Mỗi bước huấn luyện (một batch 64 ảnh) diễn ra như sau:
1. **Lan truyền tiến** qua các block theo thứ tự ở mục 2.8, ra xác suất của 7 lớp.
2. **Tính loss** bằng thư viện.
3. **Lan truyền ngược:** gradient của loss được tính qua Softmax (thư viện), rồi đi ngược qua từng block theo thứ tự Dense, Dropout, BatchNorm, ReLU, Dense, Flatten, MaxPool, Dropout, BatchNorm, ReLU, Conv... Tại mỗi block TensorFlow gọi hàm `grad` đã tự viết ở Phần 2, hàm này vừa tính gradient theo tham số của block vừa trả gradient về cho block phía trước.
4. **Adam** dùng gradient của các tham số (kernel, bias, $\gamma$, $\beta$, trọng số Dense) để cập nhật chúng.

Tham số `jit_compile=False` chỉ để tắt XLA cho các phép toán tự viết chạy ổn định, không ảnh hưởng đến kết quả huấn luyện.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),   # Adam dùng thư viện
    loss=tf.keras.losses.CategoricalCrossentropy(),            # loss dùng thư viện
    metrics=['accuracy'],
    jit_compile=False                                          # tắt XLA
)

### 3.2. Callbacks

- **ModelCheckpoint:** lưu mô hình vào file `model.h5` mỗi khi `val_accuracy` đạt giá trị cao nhất từ trước đến giờ.
- **ReduceLROnPlateau:** nếu `val_loss` không giảm trong 3 epoch liên tiếp thì nhân learning rate với 0.2, không nhỏ hơn 0.00001.

In [ ]:
checkpoint = ModelCheckpoint(
    'model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=0.00001,
    verbose=1
)

### 3.3. Huấn luyện

Huấn luyện 25 epoch, đánh giá trên validation sau mỗi epoch. Các block tích chập tự viết chạy chậm hơn bản dựng sẵn của Keras nên nên chạy trên GPU.

In [ ]:
print("Bắt đầu quá trình huấn luyện mô hình CNN.")
history = model.fit(
    train_generator,
    epochs=25,
    validation_data=validation_generator,
    callbacks=[checkpoint, reduce_lr]
)
print("Huấn luyện xong! File đã train được lưu thành 'model.h5'.")

Bắt đầu quá trình huấn luyện mô hình CNN.
Epoch 1/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.2214 - loss: 2.4270
Epoch 1: val_accuracy improved from None to 0.25309, saving model to model.h5



Epoch 1: finished saving model to model.h5
359/359 ━━━━━━━━━━━━━━━━━━━━ 41s 87ms/step - accuracy: 0.2461 - loss: 2.1608 - val_accuracy: 0.2531 - val_loss: 2.0199 - learning_rate: 0.0010
Epoch 2/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.3317 - loss: 1.7494
Epoch 2: val_accuracy improved from 0.25309 to 0.39836, saving model to model.h5



Epoch 2: finished saving model to model.h5
359/359 ━━━━━━━━━━━━━━━━━━━━ 30s 84ms/step - accuracy: 0.3485 - loss: 1.6900 - val_accuracy: 0.3984 - val_loss: 1.5471 - learning_rate: 0.0010
Epoch 3/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.4020 - loss: 1.5315
Epoch 3: val_accuracy did not improve from 0.39836
359/359 ━━━━━━━━━━━━━━━━━━━━ 29s 82ms/step - accuracy: 0.4181 - loss: 1.4997 - val_accuracy: 0.3451 - val_loss: 1.9182 - learning_rate: 0.0010
Epoch 4/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.4455 - loss: 1.4424
Epoch 4: val_accuracy improved from 0.39836 to 0.43529, saving model to model.h5



Epoch 4: finished saving model to model.h5
359/359 ━━━━━━━━━━━━━━━━━━━━ 30s 83ms/step - accuracy: 0.4545 - loss: 1.4215 - val_accuracy: 0.4353 - val_loss: 1.4363 - learning_rate: 0.0010
Epoch 5/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.4757 - loss: 1.3670
Epoch 5: val_accuracy improved from 0.43529 to 0.48197, saving model to model.h5



Epoch 5: finished saving model to model.h5
359/359 ━━━━━━━━━━━━━━━━━━━━ 30s 83ms/step - accuracy: 0.4768 - loss: 1.3656 - val_accuracy: 0.4820 - val_loss: 1.3809 - learning_rate: 0.0010
Epoch 6/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.4868 - loss: 1.3350
Epoch 6: val_accuracy did not improve from 0.48197
359/359 ━━━━━━━━━━━━━━━━━━━━ 29s 81ms/step - accuracy: 0.4912 - loss: 1.3288 - val_accuracy: 0.4261 - val_loss: 2.0059 - learning_rate: 0.0010
Epoch 7/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.4955 - loss: 1.3203
Epoch 7: val_accuracy did not improve from 0.48197
359/359 ━━━━━━━━━━━━━━━━━━━━ 29s 82ms/step - accuracy: 0.5007 - loss: 1.3124 - val_accuracy: 0.4377 - val_loss: 1.4651 - learning_rate: 0.0010
Epoch 8/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.5071 - loss: 1.2830
Epoch 8: val_accuracy improved from 0.48197 to 0.51385, saving model to model.h5



Epoch 8: finished saving model to model.h5
359/359 ━━━━━━━━━━━━━━━━━━━━ 29s 81ms/step - accuracy: 0.5103 - loss: 1.2800 - val_accuracy: 0.5138 - val_loss: 1.2854 - learning_rate: 0.0010
Epoch 9/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.5173 - loss: 1.2646
Epoch 9: val_accuracy improved from 0.51385 to 0.52569, saving model to model.h5



Epoch 9: finished saving model to model.h5
359/359 ━━━━━━━━━━━━━━━━━━━━ 31s 85ms/step - accuracy: 0.5157 - loss: 1.2709 - val_accuracy: 0.5257 - val_loss: 1.2558 - learning_rate: 0.0010
Epoch 10/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.5235 - loss: 1.2505
Epoch 10: val_accuracy did not improve from 0.52569
359/359 ━━━━━━━━━━━━━━━━━━━━ 29s 81ms/step - accuracy: 0.5253 - loss: 1.2459 - val_accuracy: 0.3708 - val_loss: 3.0650 - learning_rate: 0.0010
Epoch 11/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.5291 - loss: 1.2294
Epoch 11: val_accuracy did not improve from 0.52569
359/359 ━━━━━━━━━━━━━━━━━━━━ 29s 82ms/step - accuracy: 0.5278 - loss: 1.2364 - val_accuracy: 0.4590 - val_loss: 1.6369 - learning_rate: 0.0010
Epoch 12/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.5394 - loss: 1.2209
Epoch 12: val_accuracy did not improve from 0.52569
359/359 ━━━━━━━━━━━━━━━━━━━━ 29s 81ms/step - accuracy: 0.5376 - loss: 1.2164 - val_accuracy: 0.5246 - val_l


Epoch 13: finished saving model to model.h5
359/359 ━━━━━━━━━━━━━━━━━━━━ 30s 82ms/step - accuracy: 0.5408 - loss: 1.2130 - val_accuracy: 0.5454 - val_loss: 1.2141 - learning_rate: 0.0010
Epoch 14/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.5458 - loss: 1.1977
Epoch 14: val_accuracy did not improve from 0.54538
359/359 ━━━━━━━━━━━━━━━━━━━━ 30s 83ms/step - accuracy: 0.5472 - loss: 1.1941 - val_accuracy: 0.5017 - val_loss: 1.2972 - learning_rate: 0.0010
Epoch 15/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.5543 - loss: 1.1818
Epoch 15: val_accuracy did not improve from 0.54538
359/359 ━━━━━━━━━━━━━━━━━━━━ 29s 81ms/step - accuracy: 0.5535 - loss: 1.1844 - val_accuracy: 0.4936 - val_loss: 1.3400 - learning_rate: 0.0010
Epoch 16/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.5534 - loss: 1.1739
Epoch 16: val_accuracy did not improve from 0.54538
359/359 ━━━━━━━━━━━━━━━━━━━━ 29s 81ms/step - accuracy: 0.5540 - loss: 1.1752 - val_accuracy: 0.5443 - val_


Epoch 19: finished saving model to model.h5
359/359 ━━━━━━━━━━━━━━━━━━━━ 30s 84ms/step - accuracy: 0.5637 - loss: 1.1496 - val_accuracy: 0.5724 - val_loss: 1.1270 - learning_rate: 0.0010
Epoch 20/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.5634 - loss: 1.1516
Epoch 20: val_accuracy did not improve from 0.57237
359/359 ━━━━━━━━━━━━━━━━━━━━ 29s 81ms/step - accuracy: 0.5665 - loss: 1.1475 - val_accuracy: 0.5238 - val_loss: 1.3057 - learning_rate: 0.0010
Epoch 21/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.5662 - loss: 1.1453
Epoch 21: val_accuracy did not improve from 0.57237
359/359 ━━━━━━━━━━━━━━━━━━━━ 29s 82ms/step - accuracy: 0.5662 - loss: 1.1443 - val_accuracy: 0.5604 - val_loss: 1.1649 - learning_rate: 0.0010
Epoch 22/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.5715 - loss: 1.1296
Epoch 22: val_accuracy did not improve from 0.57237

Epoch 22: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
359/359 ━━━━━━━━━━━━━━━━━━━


Epoch 23: finished saving model to model.h5
359/359 ━━━━━━━━━━━━━━━━━━━━ 31s 87ms/step - accuracy: 0.5845 - loss: 1.1003 - val_accuracy: 0.5875 - val_loss: 1.0861 - learning_rate: 2.0000e-04
Epoch 24/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.5938 - loss: 1.0670
Epoch 24: val_accuracy did not improve from 0.58753
359/359 ━━━━━━━━━━━━━━━━━━━━ 30s 84ms/step - accuracy: 0.5901 - loss: 1.0811 - val_accuracy: 0.5868 - val_loss: 1.0963 - learning_rate: 2.0000e-04
Epoch 25/25
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.5944 - loss: 1.0738
Epoch 25: val_accuracy improved from 0.58753 to 0.58944, saving model to model.h5



Epoch 25: finished saving model to model.h5
359/359 ━━━━━━━━━━━━━━━━━━━━ 31s 86ms/step - accuracy: 0.5933 - loss: 1.0792 - val_accuracy: 0.5894 - val_loss: 1.1012 - learning_rate: 2.0000e-04
Huấn luyện xong! File đã train được lưu thành 'model.h5'.


## Phần 4. Đánh giá trên tập test

Thư mục `test/` chưa từng được dùng để huấn luyện hay để chọn mô hình tốt nhất (việc chọn dựa trên `val_accuracy` của phần validation), nên độ chính xác ở đây phản ánh khả năng tổng quát hóa thật của mô hình.

- Bộ đọc test chỉ dùng `rescale=1./255`, **không** tăng cường dữ liệu, và `shuffle=False`.
- Nạp lại `model.h5` (file lưu ở epoch có `val_accuracy` cao nhất). Vì mô hình dùng các lớp tự viết nên phải khai báo chúng qua `custom_objects` khi nạp.
- `model.evaluate` trả về loss và accuracy trên toàn bộ ảnh test.

In [ ]:
test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(48, 48),
    color_mode='grayscale',
    batch_size=64,
    class_mode='categorical',
    shuffle=False
)

custom_objects = {
    'Conv2DLayer': Conv2DLayer,
    'ReLULayer': ReLULayer,
    'BatchNormLayer': BatchNormLayer,
    'MaxPool2DLayer': MaxPool2DLayer,
    'DropoutLayer': DropoutLayer,
    'FlattenLayer': FlattenLayer,
    'DenseLayer': DenseLayer,
}

best_model = tf.keras.models.load_model('model.h5', custom_objects=custom_objects, compile=False)
best_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy'],
    jit_compile=False
)

test_loss, test_acc = best_model.evaluate(test_generator)
print(f"Test loss: {test_loss:.4f} - Test accuracy: {test_acc:.4f}")

Found 7178 images belonging to 7 classes.
113/113 ━━━━━━━━━━━━━━━━━━━━ 5s 35ms/step - accuracy: 0.6050 - loss: 1.0633
Test loss: 1.0633 - Test accuracy: 0.6050
